## 📖 Glosarium Fitur Dataset Mirai

Dataset ini adalah **Mirai Botnet Dataset** dari HuggingFace (bornpresident/mirai_botnet), berisi 1.083.206 baris network traffic capture dari serangan botnet Mirai.

### Fitur Utama (48 total)

| Fitur | Definisi | Mengapa Penting |
|-------|----------|----------------|
| `flow_duration` | Durasi network flow dalam detik | Flow sangat pendek/panjang → anomali |
| `Header_Length` | Panjang header paket | Header abnormal → packet manipulation |
| `Protocol_Type` | Tipe protokol (6=TCP, 17=UDP, 47=GRE) | GRE flood = signature Mirai |
| `Duration` | Durasi koneksi | Serangan biasanya durasi tetap |
| `Rate` | Jumlah paket per detik | Rate tinggi → flood attack |
| `Srate` | Source rate (paket dari sumber) | Asimetri → DDoS |
| `Drate` | Destination rate | Perbedaan Srate/Drate → attack |
| `syn_flag_number` | Jumlah flag TCP SYN | SYN tinggi → SYN flood |
| `fin_flag_number` | Jumlah flag TCP FIN | FIN tinggi → connection termination attack |
| `rst_flag_number` | Jumlah flag TCP RST | RST tinggi → connection reset attack |
| `psh_flag_number` | Jumlah flag TCP PSH | PSH pattern → data exfiltration |
| `ack_flag_number` | Jumlah flag TCP ACK | ACK pattern → handshake analysis |
| `HTTP/HTTPS/DNS/Telnet/SMTP/SSH/IRC` | Counter protokol aplikasi | Telnet tinggi → Mirai signature |
| `TCP/UDP/DHCP/ARP/ICMP` | Counter protokol network layer | UDP tinggi → UDP flood |
| `AVG` | Rata-rata statistik flow |Baseline perilaku normal |
| `Std` | Standard deviation flow | Variabilitas → attack pattern |
| `Variance` | Variansi statistik flow | Inconsistensi → anomali |
| `Tot_size` | Total ukuran traffic | Volume besar → flood |
| `Weight` | Bobot statistik flow | Feature importance |

### Label
- `Benign` — Traffic normal (318K records, ~29%)
- `Malicious` — Traffic serangan Mirai (765K records, ~71%)
  - `Mirai-greeth_flood` — GRE tunnel flood (288K)
  - `Mirai-udpplain` — UDP plain flood (259K)
  - `Mirai-greip_flood` — GRE-IP flood (218K)


In [ ]:
# ============================================
# SETUP: Load Mirai Botnet Dataset
# ============================================
# Dataset: https://huggingface.co/datasets/bornpresident/mirai_botnet
# 1.083.206 baris, 48 kolom, 4 kelas traffic
# Label: BenignTraffic, Mirai-greeth_flood, Mirai-udpplain, Mirai-greip_flood
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load dataset dari HuggingFace
ds = load_dataset('bornpresident/mirai_botnet')
df = ds['train'].to_pandas()

# Rename kolom agar lebih mudah dipakai (ganti spasi dengan underscore)
df.columns = [c.replace(' ', '_') for c in df.columns]

# Rename kolom 'label' jadi 'traffic_type' agar konsisten
df = df.rename(columns={'label': 'traffic_type'})

# Konversi is_benign ke format Benign/Malicious
df['traffic_type'] = df['traffic_type'].apply(
    lambda x: 'Benign' if x == 'BenignTraffic' else 'Malicious'
)

# Ambil sample 5000 baris untuk performa notebook
# (dataset penuh 1M+ baris — sample cukup untuk EDA)
np.random.seed(42)
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Traffic types:\n{df['traffic_type'].value_counts()}")
print(f"\nKolom ({len(df.columns)} total):")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col} ({df[col].dtype})")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")


## 📖 Glosarium Feature Dataset

Dataset ini adalah **data sintetis** yang di-generate untuk mensimulasikan network traffic realistis dengan 15% malicious traffic. Berikut definisi setiap feature:

| Feature | Tipe | Definisi | Rentang Normal | Mengapa Penting |
|---------|------|----------|----------------|-----------------|
| `timestamp` | datetime | Waktu capture data (interval 1 menit) | 2024-01-01 + 5000 min | Basis analisis time-series |
| `Rate` | float | Jumlah paket per detik | ~150 avg (exp) | Traffic tinggi → mungkin DDoS/scan |
| `Tot_size` | float | Jumlah byte per detik | ~50K avg (exp) | Volume data abnormal → exfiltration |
| `AVG` | float | **Host-based Deviation Profile** (mean) — seberapa jauh perilaku host dari baseline normal. 0 = normal, 1 = sangat menyimpang | 0.35–0.65 (μ=0.5, σ=0.15) | Host yang menyimpang = indikator compromise |
| `Variance` | float | **HDP Variance** — seberapa tidak konsisten deviasi host dari waktu ke waktu | ~0.02 avg (exp) | Variance tinggi → perilaku tidak stabil (malware) |
| `flow_duration` | float | **Delta HDP** (mean) — perubahan/deviasi HDP antar waktu. Mengukur **laju perubahan** perilaku host | -0.04–0.06 (μ=0.01, σ=0.05) | Perubahan cepat → host sedang ter-compromise |
| `Std` | float | **Delta HDP Variance** — konsistensi perubahan HDP | ~0.005 avg (exp) | Fluktuasi cepat → aktivitas tidak teratur |
| `Weight` | float | **Shannon Entropy** distribusi IP/port tujuan. Mengukur **keacakan** traffic. Entropy tinggi = banyak tujuan berbeda | 0.1–0.5 (β(2,5)) | Entropy tinggi → scanning, worm spreading |
| `Number` | int | Jumlah **IP sumber unik** yang terhubung | ~50 avg (Poisson) | Banyak IP → botnet atau DDoS |
| `Magnitue` | int | Jumlah **port tujuan unik** yang diakses | ~30 avg (Poisson) | Banyak port → port scanning |
| `syn_flag_number` | float | Rasio paket **SYN** terhadap total paket TCP. SYN tanpa ACK = SYN flood | 0.05–0.15 (β(1,9)) | Rasio tinggi → SYN flood attack |
| `ICMP` | float | Rasio paket **ICMP** terhadap total paket | 0.02–0.08 (β(1,20)) | ICMP tinggi → ping sweep / ICMP tunnel |
| `traffic_type` | str | Label kelas: **`Benign`** (85%) atau **`Malicious`** (15%) | - | Ground truth untuk supervised learning |

### 🔑 Feature Utama yang Sering Dipakai

**HDP (Host-based Deviation Profile):**
- Mengukur seberapa "aneh" perilaku sebuah host dibandingkan baseline normal
- `AVG` = rata-rata deviasi (nilai lebih tinggi = lebih menyimpang)
- `Variance` = seberapa tidak konsisten deviasinya
- Pada malicious traffic: AVG ~0.85 (jauh dari baseline 0.5)

**dHDP (Delta HDP):**
- Mengukur **perubahan** HDP dari waktu ke waktu (turunan/deviasi pertama)
- `flow_duration` = rata-rata perubahan (positif = semakin menyimpang)
- `Std` = seberapa fluktuatif perubahannya
- Berguna untuk mendeteksi host yang **sedang aktif** di-compromise (bukan yang sudah lama)

### 📊 Distribusi per Traffic Type
- **Benign:** 4250 samples (85%) — traffic normal
- **Malicious:** 750 samples (15%) — traffic dengan pattern serangan
- Missing values: 50 per kolom di `AVG`, `flow_duration`, `Weight` (mensimulasikan data real-world)

---
## Soal 1: Mean — Rata-rata Traffic Metrics

Hitung rata-rata dari setiap fitur numerik. Bandingkan mean antara traffic Benign vs Malicious. Fitur mana yang paling besar perbedaannya?

In [ ]:

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
mean_by_type = df.groupby('traffic_type')[numeric_cols].mean()
print("=== Mean per Traffic Type ===")
print(mean_by_type.round(4))

# Hitung perbedaan
diff = mean_by_type.loc['Malicious'] - mean_by_type.loc['Benign']
diff_pct = (diff / mean_by_type.loc['Benign'] * 100).sort_values(ascending=False)
print("\n=== % Difference (Malicious vs Benign) ===")
print(diff_pct.round(2))
print(f"\n\n📌 Fitur dengan perbedaan terbesar: {diff_pct.index[0]} ({diff_pct.iloc[0]:.1f}%)")

---
## Soal 2: Median — Nilai Tengah Traffic Volume

Hitung median untuk `Rate` dan `Tot_size`. Apakah median jauh berbeda dari mean? Apa implikasinya terhadap distribusi data?

In [ ]:

for col in ['Rate', 'Tot_size']:
    mean_val = df[col].mean()
    median_val = df[col].median()
    print(f"{col}:")
    print(f"  Mean:   {mean_val:.2f}")
    print(f"  Median: {median_val:.2f}")
    print(f"  Selisih: {abs(mean_val - median_val):.2f}")
    print(f"  Mean > Median → {'Ya (skew kanan)' if mean_val > median_val else 'Tidak'}")
    print()

---
## Soal 3: Mode — Kategori Traffic Paling Umum

Apa traffic type yang paling sering muncul (mode)? Berapa frekuensinya?

In [ ]:

mode_type = df['traffic_type'].mode()[0]
counts = df['traffic_type'].value_counts()
print(f"Mode traffic_type: {mode_type}")
print(f"\nFrekuensi:")
for t, c in counts.items():
    pct = c / len(df) * 100
    print(f"  {t}: {c} ({pct:.1f}%)")

---
## Soal 4: Variance & Standard Deviation

Hitung variance dan standard deviation dari `AVG`, `Weight`, dan `syn_flag_number`. Fitur mana yang paling stabil (std terkecil)?

In [ ]:

cols = ['AVG', 'Weight', 'syn_flag_number']
stats = pd.DataFrame({
    'Variance': df[cols].var(),
    'Std Dev': df[cols].std(),
    'CV (%)': df[cols].std() / df[cols].mean() * 100  # Coefficient of Variation
})
print(stats.round(4))
print(f"\n📌 Fitur paling stabil (std terkecil): {stats['Std Dev'].idxmin()}")
print(f"📌 Fitur paling variatif (CV tertinggi): {stats['CV (%)'].idxmax()}")

---
## Soal 5: Range — Rentang Nilai Minimum & Maximum

Tentukan range (min dan max) dari setiap fitur numerik. Apakah ada nilai yang tidak masuk akal (misal: negatif untuk metric yang harusnya positif)?

In [ ]:

range_df = pd.DataFrame({
    'Min': df[numeric_cols].min(),
    'Max': df[numeric_cols].max(),
    'Range': df[numeric_cols].max() - df[numeric_cols].min()
})
print(range_df.round(4))

# Cek nilai negatif yang tidak seharusnya
print("\n⚠️ Cek nilai negatif:")
for col in ['Rate', 'Tot_size', 'Variance', 'Std', 'Weight',
            'Number', 'Magnitue', 'syn_flag_number', 'ICMP']:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"  {col}: {neg_count} nilai negatif")
    else:
        print(f"  {col}: OK (tidak ada nilai negatif)")

---
## Soal 6: Percentiles — Threshold Anomali

Hitung percentil 25, 50, 75, dan 95 dari `Rate` dan `AVG`. Gunakan percentil 95 sebagai threshold untuk mendeteksi potensi anomali. Berapa banyak data yang melebihi threshold ini?

In [ ]:

percentiles = [25, 50, 75, 95]
for col in ['Rate', 'AVG']:
    print(f"\n{col}:")
    for p in percentiles:
        val = df[col].quantile(p / 100)
        print(f"  P{p}: {val:.4f}")
    
    threshold_95 = df[col].quantile(0.95)
    above = (df[col] > threshold_95).sum()
    malicious_above = df.loc[df[col] > threshold_95, 'traffic_type'].value_counts()
    print(f"  Data di atas P95: {above} ({above/len(df)*100:.1f}%)")
    print(f"  Komposisi: {dict(malicious_above)}")

---
## Soal 7: Quartiles & IQR — Deteksi Outlier

Hitung Q1, Q2, Q3, dan IQR untuk `Rate`. Gunakan rumus outlier: nilai < Q1 - 1.5×IQR atau > Q3 + 1.5×IQR. Berapa banyak outlier? Apakah outlier cenderung malicious?

In [ ]:

col = 'Rate'
Q1 = df[col].quantile(0.25)
Q2 = df[col].quantile(0.50)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"{col}:")
print(f"  Q1: {Q1:.2f}")
print(f"  Q2 (Median): {Q2:.2f}")
print(f"  Q3: {Q3:.2f}")
print(f"  IQR: {IQR:.2f}")
print(f"  Batas bawah: {lower_bound:.2f}")
print(f"  Batas atas: {upper_bound:.2f}")

outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
print(f"\nTotal outlier: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
print(f"\nKomposisi outlier:")
print(outliers['traffic_type'].value_counts())

mal_rate = (outliers['traffic_type'] == 'Malicious').sum() / len(outliers) * 100
print(f"\n📌 {mal_rate:.1f}% outlier adalah traffic Malicious")
print(f"📌 IQR method berhasil mendeteksi {(outliers['traffic_type'] == 'Malicious').sum()} malicious traffic dari total {(df['traffic_type'] == 'Malicious').sum()}")